# Brain Tumor Classification — DenseNet-201

Dataset: Brain Tumor MRI Dataset (Kaggle) — Google Drive er **BrainTumor** folder e rakha ache.
Ei notebook DenseNet-201 model train kore, test set e accuracy ber kore, Grad-CAM heatmap banay,
ar sob result (confusion matrix, Grad-CAM image, results file, trained model) **BrainTumor/Result**
folder e save kore dey.

Sudhu **Runtime → Run all** chaplei hobe. (Age Runtime → Change runtime type → **T4 GPU** select kore nio.)

In [ ]:
# Google Drive mount kori
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, random, shutil, zipfile, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DRIVE_FOLDER = '/content/drive/MyDrive/BrainTumor'
RESULT_FOLDER = os.path.join(DRIVE_FOLDER, 'Result')
os.makedirs(RESULT_FOLDER, exist_ok=True)
print('Result folder:', RESULT_FOLDER)

In [ ]:
# Dataset khuji - folder akare thakle direct, zip thakle unzip kore
def find_dataset(base):
    for root, dirs, files in os.walk(base):
        if 'Training' in dirs and 'Testing' in dirs:
            return root
    return None

data_root = find_dataset(DRIVE_FOLDER)

if data_root is None:
    for name in os.listdir(DRIVE_FOLDER):
        if name.endswith('.zip'):
            print('Zip peyechi, unzip kori:', name)
            with zipfile.ZipFile(os.path.join(DRIVE_FOLDER, name)) as z:
                z.extractall('/content/brain_data')
            data_root = find_dataset('/content/brain_data')
            break

print('Dataset path:', data_root)

classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
for split in ['Training', 'Testing']:
    for c in classes:
        n = len(os.listdir(os.path.join(data_root, split, c)))
        print(split, '-', c, ':', n)

In [ ]:
# Training data ke train / validation e vag kori (seed 42, 12% validation)
# Test set alada-i thake, oita sudhu seshe ekbar evaluate hobe
rng = random.Random(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

train_items, val_items, test_items = [], [], []
for label, c in enumerate(classes):
    folder = os.path.join(data_root, 'Training', c)
    files = [os.path.join(folder, f) for f in sorted(os.listdir(folder))]
    rng.shuffle(files)
    n_val = int(round(len(files) * 0.12))
    val_items += [(f, label) for f in files[:n_val]]
    train_items += [(f, label) for f in files[n_val:]]

    folder = os.path.join(data_root, 'Testing', c)
    test_items += [(os.path.join(folder, f), label) for f in sorted(os.listdir(folder))]

print('Train:', len(train_items))
print('Validation:', len(val_items))
print('Test:', len(test_items))

In [ ]:
# Image preprocessing + augmentation (augmentation sudhu training e)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class BrainDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

train_loader = DataLoader(BrainDataset(train_items, train_transform), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(BrainDataset(val_items, test_transform), batch_size=64, num_workers=2)
test_loader = DataLoader(BrainDataset(test_items, test_transform), batch_size=64, num_workers=2)
print('Loaders ready')

In [ ]:
# DenseNet-201 model (ImageNet pretrained) - last layer ta 4 class er jonno change kori
model = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 4)
)
model = model.to(device)
print(model.classifier)

In [ ]:
# Ekta loader er upor accuracy, macro F1 ar confusion matrix ber korar function
def evaluate(loader):
    model.eval()
    cm = np.zeros((4, 4), dtype=int)
    with torch.no_grad():
        for x, y in loader:
            pred = model(x.to(device)).argmax(1).cpu().numpy()
            for t, p in zip(y.numpy(), pred):
                cm[t][p] += 1
    acc = np.trace(cm) / cm.sum()
    f1_list = []
    for i in range(4):
        tp = cm[i][i]
        precision = tp / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
        recall = tp / cm[i, :].sum() if cm[i, :].sum() > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_list.append(f1)
    return acc, float(np.mean(f1_list)), cm

In [ ]:
# Training
epochs = 20
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

best_f1 = 0
train_loss_history = []
val_acc_history = []

for epoch in range(epochs):
    start = time.time()
    model.train()
    total_loss, total = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        total += len(y)
    scheduler.step()

    val_acc, val_f1, _ = evaluate(val_loader)
    train_loss_history.append(total_loss / total)
    val_acc_history.append(val_acc)
    print(f'Epoch {epoch+1}/{epochs}  loss={total_loss/total:.4f}  val_acc={val_acc*100:.2f}%  val_f1={val_f1*100:.2f}%  ({time.time()-start:.0f}s)')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pth')
        print('   -> best model saved')

print('Training sesh. Best validation F1:', round(best_f1 * 100, 2), '%')

In [ ]:
# Final test - best model diye
model.load_state_dict(torch.load('best_model.pth'))
test_acc, test_f1, cm = evaluate(test_loader)

print('=' * 45)
print('TEST ACCURACY :', round(test_acc * 100, 2), '%')
print('TEST MACRO F1 :', round(test_f1 * 100, 2), '%')
print('=' * 45)
print()
print('Class wise result:')
for i, c in enumerate(classes):
    tp = cm[i][i]
    precision = tp / cm[:, i].sum() * 100
    recall = tp / cm[i, :].sum() * 100
    f1 = 2 * precision * recall / (precision + recall)
    print(f'{c:12s} Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}  Total={cm[i, :].sum()}')
print()
print('Confusion Matrix (row = asol class, column = prediction):')
print(cm)
print('Vul prediction:', int(cm.sum() - np.trace(cm)), '/', int(cm.sum()))

In [ ]:
# Confusion matrix er chobi
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - DenseNet-201')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i][j], ha='center', va='center',
                color='white' if cm[i][j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'confusion_matrix.png'), dpi=150)
plt.show()

# Training curve er chobi
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(train_loss_history, color='red', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='red')
ax2 = ax1.twinx()
ax2.plot([a * 100 for a in val_acc_history], color='blue', label='Val Accuracy %')
ax2.set_ylabel('Val Accuracy %', color='blue')
plt.title('Training Curve - DenseNet-201')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'training_curve.png'), dpi=150)
plt.show()

In [ ]:
# Grad-CAM: model image er kon jayga dekhe decision nilo seta heatmap diye dekhai
# Last convolution block er output theke activation + gradient niye heatmap banano hoy
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, x, class_idx):
        self.model.zero_grad()
        output = self.model(x)
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = torch.relu(cam)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

In [ ]:
# Prottek class theke ekta kore test image niye Grad-CAM heatmap banai
# Lal/holud ongsho = model oi jayga dekhei decision niyeche
target_layer = list(model.features.children())[-2]
gradcam = GradCAM(model, target_layer)
model.eval()

picked = []
for label in range(4):
    for path, y in test_items:
        if y == label:
            picked.append((path, y))
            break

fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for row, (path, label) in enumerate(picked):
    image = Image.open(path).convert('RGB')
    x = test_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    pred = int(probs.argmax())
    conf = float(probs[pred]) * 100

    cam = gradcam.generate(x, pred)
    cam_big = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224))) / 255.0
    heatmap = plt.cm.jet(cam_big)[:, :, :3]
    original = np.array(image.resize((224, 224))) / 255.0
    overlay = 0.55 * heatmap + 0.45 * original

    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f'Original: {classes[label]}')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(cam_big, cmap='jet')
    axes[row, 1].set_title('Grad-CAM Heatmap')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Predicted: {classes[pred]} ({conf:.1f}%)')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULT_FOLDER, 'gradcam_examples.png'), dpi=150)
plt.show()
print('Grad-CAM image save hoyeche: gradcam_examples.png')

In [ ]:
# Sob result Drive er Result folder e save kori
lines = []
lines.append('Brain Tumor Classification - DenseNet-201')
lines.append('Dataset: Brain Tumor MRI Dataset (Kaggle, original split)')
lines.append('')
lines.append(f'Test Accuracy : {test_acc * 100:.2f}%')
lines.append(f'Test Macro F1 : {test_f1 * 100:.2f}%')
lines.append('')
lines.append('Class wise (Precision / Recall / F1 / Total):')
for i, c in enumerate(classes):
    tp = cm[i][i]
    precision = tp / cm[:, i].sum() * 100
    recall = tp / cm[i, :].sum() * 100
    f1 = 2 * precision * recall / (precision + recall)
    lines.append(f'{c:12s} {precision:.2f} / {recall:.2f} / {f1:.2f} / {cm[i, :].sum()}')
lines.append('')
lines.append('Confusion Matrix:')
lines.append(str(cm))
lines.append('')
lines.append(f'Errors: {int(cm.sum() - np.trace(cm))} / {int(cm.sum())}')

with open(os.path.join(RESULT_FOLDER, 'results.txt'), 'w') as f:
    f.write('\n'.join(lines))

shutil.copy('best_model.pth', os.path.join(RESULT_FOLDER, 'best_model.pth'))

print('Save hoyeche:')
for name in sorted(os.listdir(RESULT_FOLDER)):
    print(' -', name)